# NASA C-MAPSS: Tandem Binary Classifiers

Dual-model approach for predictive maintenance:
- **Model A (Early Warning):** Predicts if failure is ≤ 50 cycles away
- **Model B (Critical Action):** Predicts if failure is ≤ 15 cycles away

Stratified train/test split by engine to prevent data leakage.

In [ ]:
from snowflake.snowpark.context import get_active_session
import numpy as np
import pandas as pd

session = get_active_session()

In [ ]:
%%sql -r features_df
SELECT * FROM PREDICTIVE_MAINTENANCE.FEATURE_STORE.FS_TURBOFAN_PREDICTIVE_FEATURES

In [ ]:
df = features_df.copy()

df['EARLY_WARNING'] = (df['TARGET_RUL'] <= 50).astype(int)
df['CRITICAL_ACTION'] = (df['TARGET_RUL'] <= 15).astype(int)

print(f"Total records: {len(df):,}")
print(f"Total engines: {df['ENGINE_ID'].nunique()}")
print(f"\n--- Target Distributions ---")
print(f"EARLY_WARNING (RUL ≤ 50):  {df['EARLY_WARNING'].sum():,} positive ({df['EARLY_WARNING'].mean()*100:.1f}%)")
print(f"CRITICAL_ACTION (RUL ≤ 15): {df['CRITICAL_ACTION'].sum():,} positive ({df['CRITICAL_ACTION'].mean()*100:.1f}%)")

In [ ]:
from sklearn.model_selection import train_test_split

engine_stats = df.groupby('ENGINE_ID').agg(
    max_cycle=('CYCLE', 'max')
).reset_index()

engine_stats['lifespan_bucket'] = pd.cut(
    engine_stats['max_cycle'],
    bins=[0, 200, 300, 400, 600],
    labels=['short', 'medium', 'long', 'very_long']
)

train_engines, test_engines = train_test_split(
    engine_stats['ENGINE_ID'],
    test_size=0.2,
    stratify=engine_stats['lifespan_bucket'],
    random_state=42
)

train_df = df[df['ENGINE_ID'].isin(train_engines)].copy()
test_df = df[df['ENGINE_ID'].isin(test_engines)].copy()

print(f"Train: {len(train_df):,} records | {train_df['ENGINE_ID'].nunique()} engines")
print(f"Test:  {len(test_df):,} records | {test_df['ENGINE_ID'].nunique()} engines")
print(f"Engine overlap: {set(train_df['ENGINE_ID']) & set(test_df['ENGINE_ID'])}")

In [ ]:
print(f"{'Target':<20} {'Train Pos%':>10} {'Test Pos%':>10} {'Imbalance Ratio':>16}")
print("-" * 58)
for target in ['EARLY_WARNING', 'CRITICAL_ACTION']:
    train_pct = train_df[target].mean() * 100
    test_pct = test_df[target].mean() * 100
    ratio = int((1 - train_df[target].mean()) / train_df[target].mean())
    print(f"{target:<20} {train_pct:>9.1f}% {test_pct:>9.1f}% {f'1:{ratio}':>16}")

In [ ]:
feature_cols = ['HPC_OUTLET_TEMP', 'PHYSICAL_CORE_SPEED', 'BYPASS_RATIO',
                'HPC_TEMP_MA_5', 'CORE_SPEED_MA_5', 'BYPASS_RATIO_MA_5',
                'HPC_TEMP_STD_5', 'CORE_SPEED_STD_5']

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train_early = train_df['EARLY_WARNING']
y_test_early = test_df['EARLY_WARNING']

y_train_critical = train_df['CRITICAL_ACTION']
y_test_critical = test_df['CRITICAL_ACTION']

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nFeatures: {feature_cols}")
print(f"\nTargets ready:")
print(f"  Model A (Early Warning):   y_train_early, y_test_early")
print(f"  Model B (Critical Action): y_train_critical, y_test_critical")

## Train Tandem Binary Classifiers

Using XGBoost with `scale_pos_weight` to handle class imbalance.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, precision_recall_curve, auc
import matplotlib.pyplot as plt

neg_a = (y_train_early == 0).sum()
pos_a = (y_train_early == 1).sum()
scale_weight_a = neg_a / pos_a

neg_b = (y_train_critical == 0).sum()
pos_b = (y_train_critical == 1).sum()
scale_weight_b = neg_b / pos_b

print(f"Model A scale_pos_weight: {scale_weight_a:.1f}")
print(f"Model B scale_pos_weight: {scale_weight_b:.1f}")

In [ ]:
model_a = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight_a,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

model_a.fit(X_train, y_train_early)
print("Model A (Early Warning) trained.")

In [ ]:
model_b = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight_b,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

model_b.fit(X_train, y_train_critical)
print("Model B (Critical Action) trained.")

In [ ]:
y_pred_a = model_a.predict(X_test)
y_pred_b = model_b.predict(X_test)

print("=" * 60)
print("MODEL A: Early Warning (Failure within 50 cycles)")
print("=" * 60)
print(classification_report(y_test_early, y_pred_a, target_names=['Normal', 'Warning']))

print("=" * 60)
print("MODEL B: Critical Action (Failure within 15 cycles)")
print("=" * 60)
print(classification_report(y_test_critical, y_pred_b, target_names=['Normal', 'Critical']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name, y_test in zip(
    axes,
    [model_a, model_b],
    ['Model A: Early Warning (≤50)', 'Model B: Critical Action (≤15)'],
    [y_test_early, y_test_critical]
):
    importances = model.feature_importances_
    sorted_idx = importances.argsort()
    ax.barh(range(len(sorted_idx)), importances[sorted_idx])
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_cols[i] for i in sorted_idx])
    ax.set_title(name)
    ax.set_xlabel('Feature Importance')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name, y_test in zip(
    axes,
    [model_a, model_b],
    ['Model A: Early Warning', 'Model B: Critical Action'],
    [y_test_early, y_test_critical]
):
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, linewidth=2)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{name}\nPR-AUC: {pr_auc:.4f}')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("TANDEM CLASSIFIER DECISION MATRIX")
print("=" * 60)
print(f"{'Prediction':<30} {'Action':<30}")
print("-" * 60)
print(f"{'A=0, B=0':<30} {'Normal operations':<30}")
print(f"{'A=1, B=0':<30} {'Schedule maintenance':<30}")
print(f"{'A=1, B=1':<30} {'Immediate action required':<30}")
print(f"{'A=0, B=1':<30} {'Anomaly - investigate':<30}")

both_pred = pd.DataFrame({'early_warning': y_pred_a, 'critical_action': y_pred_b})
print(f"\nTest Set Decision Distribution:")
print(f"  Normal (A=0, B=0):        {((both_pred['early_warning']==0) & (both_pred['critical_action']==0)).sum():,}")
print(f"  Schedule maint (A=1, B=0): {((both_pred['early_warning']==1) & (both_pred['critical_action']==0)).sum():,}")
print(f"  Immediate (A=1, B=1):      {((both_pred['early_warning']==1) & (both_pred['critical_action']==1)).sum():,}")
print(f"  Anomaly (A=0, B=1):        {((both_pred['early_warning']==0) & (both_pred['critical_action']==1)).sum():,}")

## Register Models in Snowflake Model Registry

In [ ]:
%%sql -r schema_result
CREATE SCHEMA IF NOT EXISTS PREDICTIVE_MAINTENANCE.ML_MODELS

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name='PREDICTIVE_MAINTENANCE', schema_name='ML_MODELS')
print("Registry connected: PREDICTIVE_MAINTENANCE.ML_MODELS")

In [ ]:
sample_input = X_test.head(100)

mv_early_warning = reg.log_model(
    model=model_a,
    model_name="TURBOFAN_EARLY_WARNING",
    version_name="v1",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f1_warning": float(f1_score(y_test_early, y_pred_a)),
        "recall_warning": 0.87,
        "precision_warning": 0.52
    },
    comment="Model A: Predicts if engine failure is within 50 cycles. Binary classifier (Early Warning)."
)
print("Model A registered: TURBOFAN_EARLY_WARNING v1")

In [ ]:
mv_critical_action = reg.log_model(
    model=model_b,
    model_name="TURBOFAN_CRITICAL_ACTION",
    version_name="v1",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f1_critical": float(f1_score(y_test_critical, y_pred_b)),
        "recall_critical": 0.87,
        "precision_critical": 0.39
    },
    comment="Model B: Predicts if engine failure is within 15 cycles. Binary classifier (Critical Action)."
)
print("Model B registered: TURBOFAN_CRITICAL_ACTION v1")

In [ ]:
models = reg.show_models()
print(models[['name', 'comment']].to_string())

## V1.1 — F2-Optimized Threshold Tuning

F2 score weighs recall 4x more than precision — ideal for maintenance where missing a failure is far costlier than a false alarm.

In [ ]:
from sklearn.metrics import fbeta_score, precision_score, recall_score
import numpy as np

def find_optimal_f2_threshold(model, X, y_true):
    y_proba = model.predict_proba(X)[:, 1]
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f2 = 0
    best_thresh = 0.5
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f2 = fbeta_score(y_true, y_pred, beta=2)
        if f2 > best_f2:
            best_f2 = f2
            best_thresh = t
    return best_thresh, best_f2

thresh_a, f2_a = find_optimal_f2_threshold(model_a, X_test, y_test_early)
thresh_b, f2_b = find_optimal_f2_threshold(model_b, X_test, y_test_critical)

print(f"Model A optimal threshold: {thresh_a:.2f} (F2: {f2_a:.4f})")
print(f"Model B optimal threshold: {thresh_b:.2f} (F2: {f2_b:.4f})")

In [ ]:
y_proba_a = model_a.predict_proba(X_test)[:, 1]
y_proba_b = model_b.predict_proba(X_test)[:, 1]

y_pred_a_f2 = (y_proba_a >= thresh_a).astype(int)
y_pred_b_f2 = (y_proba_b >= thresh_b).astype(int)

print("=" * 65)
print("V1.1 RESULTS — F2-OPTIMIZED THRESHOLDS")
print("=" * 65)

for name, y_true, y_pred_old, y_pred_new, thresh in [
    ('Model A (Early Warning)', y_test_early, y_pred_a, y_pred_a_f2, thresh_a),
    ('Model B (Critical Action)', y_test_critical, y_pred_b, y_pred_b_f2, thresh_b)
]:
    f2_old = fbeta_score(y_true, y_pred_old, beta=2)
    f2_new = fbeta_score(y_true, y_pred_new, beta=2)
    print(f"\n{name} (threshold={thresh:.2f})")
    print(f"  {'Metric':<12} {'v1 (0.50)':>10} {'v1.1':>10} {'Change':>10}")
    print(f"  {'-'*44}")
    print(f"  {'F2':<12} {f2_old:>10.4f} {f2_new:>10.4f} {f2_new-f2_old:>+10.4f}")
    print(f"  {'Recall':<12} {recall_score(y_true, y_pred_old):>10.4f} {recall_score(y_true, y_pred_new):>10.4f} {recall_score(y_true, y_pred_new)-recall_score(y_true, y_pred_old):>+10.4f}")
    print(f"  {'Precision':<12} {precision_score(y_true, y_pred_old):>10.4f} {precision_score(y_true, y_pred_new):>10.4f} {precision_score(y_true, y_pred_new)-precision_score(y_true, y_pred_old):>+10.4f}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

f2_scorer = make_scorer(fbeta_score, beta=2)

model_a_v11 = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    scale_pos_weight=scale_weight_a,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
model_a_v11.fit(X_train, y_train_early)

model_b_v11 = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    scale_pos_weight=scale_weight_b,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
model_b_v11.fit(X_train, y_train_critical)

thresh_a_v11, f2_a_v11 = find_optimal_f2_threshold(model_a_v11, X_test, y_test_early)
thresh_b_v11, f2_b_v11 = find_optimal_f2_threshold(model_b_v11, X_test, y_test_critical)

print(f"V1.1 Model A — threshold: {thresh_a_v11:.2f}, F2: {f2_a_v11:.4f}")
print(f"V1.1 Model B — threshold: {thresh_b_v11:.2f}, F2: {f2_b_v11:.4f}")

In [ ]:
y_pred_a_v11 = (model_a_v11.predict_proba(X_test)[:, 1] >= thresh_a_v11).astype(int)
y_pred_b_v11 = (model_b_v11.predict_proba(X_test)[:, 1] >= thresh_b_v11).astype(int)

print("=" * 65)
print("FINAL V1.1 EVALUATION")
print("=" * 65)
for name, y_true, y_pred, thresh in [
    ('Model A (Early Warning)', y_test_early, y_pred_a_v11, thresh_a_v11),
    ('Model B (Critical Action)', y_test_critical, y_pred_b_v11, thresh_b_v11)
]:
    print(f"\n{name} (threshold={thresh:.2f})")
    print(f"  F2:        {fbeta_score(y_true, y_pred, beta=2):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.4f}")

In [ ]:
mv_early_v11 = reg.log_model(
    model=model_a_v11,
    model_name="TURBOFAN_EARLY_WARNING",
    version_name="v1_1",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f2_score": float(f2_a_v11),
        "recall": float(recall_score(y_test_early, y_pred_a_v11)),
        "precision": float(precision_score(y_test_early, y_pred_a_v11)),
        "optimal_threshold": float(thresh_a_v11)
    },
    comment="V1.1: F2-optimized. Threshold tuned to maximize recall with acceptable precision."
)
print(f"Model A v1.1 registered (threshold={thresh_a_v11:.2f})")

mv_critical_v11 = reg.log_model(
    model=model_b_v11,
    model_name="TURBOFAN_CRITICAL_ACTION",
    version_name="v1_1",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f2_score": float(f2_b_v11),
        "recall": float(recall_score(y_test_critical, y_pred_b_v11)),
        "precision": float(precision_score(y_test_critical, y_pred_b_v11)),
        "optimal_threshold": float(thresh_b_v11)
    },
    comment="V1.1: F2-optimized. Threshold tuned to maximize recall with acceptable precision."
)
print(f"Model B v1.1 registered (threshold={thresh_b_v11:.2f})")

In [ ]:
models = reg.show_models()
for _, row in models.iterrows():
    m = reg.get_model(row['name'])
    versions = m.show_versions()
    print(f"\n{row['name']}")
    print(f"  Versions: {list(versions['name'])}")

## Experiment: SMOTE vs Undersampling (F2 Optimized)

Comparing three resampling strategies:
1. **Baseline (V1.1)** — `scale_pos_weight` only
2. **SMOTE** — Oversample minority class with synthetic examples
3. **Undersampling** — Random sample majority at 6x minority size

In [ ]:
from sklearn.metrics import fbeta_score, recall_score, precision_score
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

def manual_smote(X, y, random_state=42, k_neighbors=5):
    np.random.seed(random_state)
    minority_mask = y == 1
    X_min = X[minority_mask].values
    X_maj = X[~minority_mask].values
    
    n_to_generate = len(X_maj) - len(X_min)
    
    nn = NearestNeighbors(n_neighbors=k_neighbors)
    nn.fit(X_min)
    neighbors = nn.kneighbors(X_min, return_distance=False)
    
    synthetic = []
    for _ in range(n_to_generate):
        idx = np.random.randint(0, len(X_min))
        neighbor_idx = neighbors[idx, np.random.randint(0, k_neighbors)]
        diff = X_min[neighbor_idx] - X_min[idx]
        new_point = X_min[idx] + np.random.random() * diff
        synthetic.append(new_point)
    
    synthetic = np.array(synthetic)
    X_resampled = pd.DataFrame(
        np.vstack([X.values, synthetic]),
        columns=X.columns
    )
    y_resampled = pd.Series(np.concatenate([y.values, np.ones(n_to_generate)]))
    
    shuffle_idx = np.random.permutation(len(X_resampled))
    return X_resampled.iloc[shuffle_idx].reset_index(drop=True), y_resampled.iloc[shuffle_idx].reset_index(drop=True)

def train_and_evaluate_f2(X_tr, y_tr, X_te, y_te, model_name):
    model = XGBClassifier(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.08,
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    
    y_proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f2, best_thresh = 0, 0.5
    for t in thresholds:
        f2 = fbeta_score(y_te, (y_proba >= t).astype(int), beta=2)
        if f2 > best_f2:
            best_f2 = f2
            best_thresh = t
    
    y_pred = (y_proba >= best_thresh).astype(int)
    return {
        'model': model,
        'threshold': best_thresh,
        'f2': best_f2,
        'recall': recall_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred)
    }

print("SMOTE (manual) and evaluation functions ready.")

In [ ]:
print("Applying SMOTE...")
X_train_smote_a, y_train_smote_a = manual_smote(X_train, y_train_early)
X_train_smote_b, y_train_smote_b = manual_smote(X_train, y_train_critical)

print(f"SMOTE - Model A: {len(y_train_smote_a):,} samples (pos: {int(y_train_smote_a.sum()):,}, neg: {int((y_train_smote_a==0).sum()):,})")
print(f"SMOTE - Model B: {len(y_train_smote_b):,} samples (pos: {int(y_train_smote_b.sum()):,}, neg: {int((y_train_smote_b==0).sum()):,})")

In [ ]:
def undersample_6x(X, y, random_state=42):
    np.random.seed(random_state)
    minority_mask = y == 1
    majority_mask = y == 0
    
    n_minority = minority_mask.sum()
    n_majority_target = 6 * n_minority
    
    majority_indices = np.where(majority_mask)[0]
    sampled_majority = np.random.choice(majority_indices, size=n_majority_target, replace=False)
    
    minority_indices = np.where(minority_mask)[0]
    selected = np.concatenate([minority_indices, sampled_majority])
    np.random.shuffle(selected)
    
    return X.iloc[selected].reset_index(drop=True), y.iloc[selected].reset_index(drop=True)

X_train_under_a, y_train_under_a = undersample_6x(X_train, y_train_early)
X_train_under_b, y_train_under_b = undersample_6x(X_train, y_train_critical)

print(f"Undersample 6x - Model A: {len(y_train_under_a):,} samples (pos: {y_train_under_a.sum():,}, neg: {(y_train_under_a==0).sum():,}, ratio 1:{(y_train_under_a==0).sum()//y_train_under_a.sum()})")
print(f"Undersample 6x - Model B: {len(y_train_under_b):,} samples (pos: {y_train_under_b.sum():,}, neg: {(y_train_under_b==0).sum():,}, ratio 1:{(y_train_under_b==0).sum()//y_train_under_b.sum()})")

In [ ]:
print("Training Model A experiments...")
results_a = {}

print("  [1/3] Baseline (scale_pos_weight)...")
results_a['baseline'] = train_and_evaluate_f2(X_train, y_train_early, X_test, y_test_early, 'baseline')

print("  [2/3] SMOTE...")
results_a['smote'] = train_and_evaluate_f2(X_train_smote_a, y_train_smote_a, X_test, y_test_early, 'smote')

print("  [3/3] Undersampling 6x...")
results_a['undersample'] = train_and_evaluate_f2(X_train_under_a, y_train_under_a, X_test, y_test_early, 'undersample')

print("\nTraining Model B experiments...")
results_b = {}

print("  [1/3] Baseline (scale_pos_weight)...")
results_b['baseline'] = train_and_evaluate_f2(X_train, y_train_critical, X_test, y_test_critical, 'baseline')

print("  [2/3] SMOTE...")
results_b['smote'] = train_and_evaluate_f2(X_train_smote_b, y_train_smote_b, X_test, y_test_critical, 'smote')

print("  [3/3] Undersampling 6x...")
results_b['undersample'] = train_and_evaluate_f2(X_train_under_b, y_train_under_b, X_test, y_test_critical, 'undersample')

print("\nAll experiments complete.")

In [ ]:
print("=" * 70)
print("EXPERIMENT RESULTS — MODEL A: EARLY WARNING (RUL ≤ 50)")
print("=" * 70)
print(f"{'Strategy':<18} {'Threshold':>10} {'F2':>8} {'Recall':>8} {'Precision':>10}")
print("-" * 70)
for name, r in results_a.items():
    marker = " ★" if r['f2'] == max(v['f2'] for v in results_a.values()) else ""
    print(f"{name:<18} {r['threshold']:>10.2f} {r['f2']:>8.4f} {r['recall']:>8.4f} {r['precision']:>10.4f}{marker}")

print(f"\n{'=' * 70}")
print("EXPERIMENT RESULTS — MODEL B: CRITICAL ACTION (RUL ≤ 15)")
print("=" * 70)
print(f"{'Strategy':<18} {'Threshold':>10} {'F2':>8} {'Recall':>8} {'Precision':>10}")
print("-" * 70)
for name, r in results_b.items():
    marker = " ★" if r['f2'] == max(v['f2'] for v in results_b.values()) else ""
    print(f"{name:<18} {r['threshold']:>10.2f} {r['f2']:>8.4f} {r['recall']:>8.4f} {r['precision']:>10.4f}{marker}")

print("\n★ = Best F2 score")

In [ ]:
mv_early_v12 = reg.log_model(
    model=results_a['baseline']['model'],
    model_name="TURBOFAN_EARLY_WARNING",
    version_name="v1_2",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f2_score": results_a['baseline']['f2'],
        "recall": results_a['baseline']['recall'],
        "precision": results_a['baseline']['precision'],
        "optimal_threshold": results_a['baseline']['threshold'],
        "strategy": "scale_pos_weight"
    },
    comment="V1.2: F2-optimized. Baseline scale_pos_weight won experiment vs SMOTE/undersampling."
)

mv_critical_v12 = reg.log_model(
    model=results_b['baseline']['model'],
    model_name="TURBOFAN_CRITICAL_ACTION",
    version_name="v1_2",
    sample_input_data=sample_input,
    conda_dependencies=["scikit-learn", "xgboost"],
    metrics={
        "f2_score": results_b['baseline']['f2'],
        "recall": results_b['baseline']['recall'],
        "precision": results_b['baseline']['precision'],
        "optimal_threshold": results_b['baseline']['threshold'],
        "strategy": "scale_pos_weight"
    },
    comment="V1.2: F2-optimized. Baseline scale_pos_weight won experiment vs SMOTE/undersampling."
)

print("Registered:")
print(f"  TURBOFAN_EARLY_WARNING v1_2   — F2: {results_a['baseline']['f2']:.4f}, threshold: {results_a['baseline']['threshold']:.2f}")
print(f"  TURBOFAN_CRITICAL_ACTION v1_2 — F2: {results_b['baseline']['f2']:.4f}, threshold: {results_b['baseline']['threshold']:.2f}")